# Analyze Synthetic Data with Heartwood on Terra

This notebook is part of the Heartwood open-source project. Copyright and license metadata are declared in `REUSE.toml`.

This notebook exercises the synthetic Heartwood reference analysis in a Terra-style Jupyter project. It uses the same persisted session as the CLI and browser interface, keeps all data synthetic, and does not claim real Terra identity binding, BigQuery access, controlled-data validation, or institutional approval. Use the no-weight `ghcr.io/schmiedmayerlab/heartwood:0.2.0-beta.2-terra` image for portable CPU use or `ghcr.io/schmiedmayerlab/heartwood:0.2.0-beta.2-terra-gpu-nvidia` for interactive local-model work, as documented in `docs/terra-jupyter-demo.md`.


## Start Heartwood

The image contains no model weights. Configure either an institution-authorized provider, a recommended local model, or another supported Hugging Face model as described in `docs/terra-jupyter-demo.md`. On a 16 GB NVIDIA GPU, the recommended local choice is `qwen25-coder-7b-instruct-awq-vllm`; Heartwood applies its 32,768-token context and checks memory and CUDA compatibility during launch. The directory containing this notebook is the Heartwood project. Open a terminal in this directory and start the browser interface. Use `heartwood launch --web` for a downloaded local model or `heartwood serve` for a hosted model:

```bash
heartwood launch --web
```

Keep that terminal running. The first code cell displays the full authenticated `/proxy/<Google project>/<cluster>/jupyter/proxy/8767/` Heartwood link for this Terra runtime. The pull-request integration uses a deterministic loopback fixture through the real OpenHands SDK and makes no model-quality claim.


In [ ]:
import json
from html import escape
from pathlib import Path
from shutil import copy2

from IPython.display import HTML
from IPython.display import display as ipython_display

from heartwood.notebook import (
    NotebookSession,
    build_widget_spec,
    jupyter_proxy_url,
)

project_root = Path.cwd().resolve()
session_id = "terra-demo"
input_root = project_root / "input"
fixture_root = Path("/opt/heartwood/fixtures/synthetic/omop-like")
input_root.mkdir(parents=True, exist_ok=True)
for filename in ("person.csv", "condition_occurrence.csv"):
    copy2(fixture_root / filename, input_root / filename)

session = NotebookSession(session_id=session_id)
proxy_url = jupyter_proxy_url(port=8767)
link = (
    f'<a href="{escape(proxy_url, quote=True)}" target="_blank" '
    'rel="noopener noreferrer">Open Heartwood in a new tab</a>'
)
ipython_display(HTML(link))
print("Heartwood project:", project_root)
print("Localized inputs:", sorted(path.name for path in input_root.iterdir()))

In [ ]:
detection = session.detect()

print("Dataset proposals")
for proposal in detection.dataset_proposals:
    print(f"- {proposal.dataset_type} ({proposal.confidence:.2f})")
    for item in proposal.evidence:
        print(f"  - {item}")

print("Latest activity:", detection.activity[-1])

In [ ]:
prompt = (
    "Build the synthetic target-condition cohort for concept 201826 with the "
    "repository-verified cohort Skill. Read the localized tables in input, require "
    "age 18 or older, apply an aggregate count floor of 20, write "
    "cohort-summary.json, and report quality checks without row-level values."
)
run = session.run(prompt)

print("Policy decisions")
for policy in run.policy_status:
    print(f"- {policy.decision}: {policy.endpoint} ({policy.reason})")

print("Approval controls")
for control in run.approval_controls:
    print(f"- {control.target_type}: {control.target_id} [{control.decision or 'pending'}]")

print("Activity")
for item in run.activity:
    print(f"{item.sequence:03d} {item.kind}: {item.detail}")

## Review And Allow The Synthetic Action

Review every member of the pending OpenHands action set in the web conversation or CLI before running the next cell. Select **Allow all once** only when the complete set invokes the repository-verified cohort script, reads `input`, writes `cohort-summary.json` inside this project, and does not transmit data.


In [ ]:
pending = [control for control in run.approval_controls if control.decision is None]
if len(pending) != 1:
    raise RuntimeError(f"Expected one pending synthetic action, found {len(pending)}")
completed = session.approve(tool_call_id=pending[0].target_id)

cohort_path = project_root / "cohort-summary.json"
cohort = json.loads(cohort_path.read_text(encoding="utf-8"))
assert cohort["summary"]["source_participant_count"] == 24
assert cohort["summary"]["source_condition_occurrence_count"] == 39
assert cohort["summary"]["participant_count"] == 20
assert cohort["summary"]["condition_occurrence_count"] == 35
assert cohort["quality_checks"]["row_values_exported"] is False
print(json.dumps(cohort, indent=2, sort_keys=True))

In [ ]:
exported = session.audit_export()
replayed = session.replay()

print("Persisted events:", replayed.event_count)
print("Exports")
for action in exported.export_actions:
    print(f"- {action.label}: {action.path}")

print("Notebook widget sections:", [section.title for section in build_widget_spec(replayed)])

## Verify the Shared Session

The CLI and browser interface read the same persisted events. Open the `terra-demo` session in the browser, then run this command from a Jupyter terminal:

```bash
heartwood --session-id terra-demo replay
```

The notebook, CLI, and browser interface should show the same researcher task, route decision, proposed terminal action, approval, successful tool outcome, and final model response. Use the interfaces sequentially for one session; independently running writers to the same file-backed session are not supported.
